# **Контрфактуалы в тексте, изображениях и рядах: Практика**

**Привет, друзья!**

Тамару все-таки взяли: теперь она ML-инженер и сама объясняет решения моделей контрфактуалами. В уроке «Расширение по модальностям: MiCE, Polyjuice, DiME, Native-Guide» мы видели, что постановка «валидность + близость + правдоподобность» переносится из таблиц в текст, картинки и ряды, а меняются только операция «минимального изменения» и модель правдоподобности. Здесь мы проверим это кодом на трех открытых наборах.

В этой практике вы:
- напишете пословное расстояние Левенштейна, которым MiCE меряет минимальность правки;
- соберете MiCE на DistilBERT SST-2 и `t5-small`: дообучите редактор восстанавливать слова при заданной метке, затем найдете бинарным поиском минимальную маску по градиенту, которая переворачивает прогноз;
- сгенерируете варианты отзывов моделью Polyjuice, отфильтруете их по беглости GPT-2 и по прогнозу классификатора и сравните с MiCE;
- найдете замены регионов по Goyal и соавторам на признаковой карте ResNet-50 для пары «полосатая кошка и египетская кошка» и сравните жадный выбор со случайным;
- обучите классификатор рядов FCN на GunPoint и построите Native-Guide: ближайший ряд другого класса, CAM вдоль времени и самый короткий переписанный подотрезок.

Табличные методы модуля собраны в тетради `PRACTICE_recourse`; здесь только текст, изображения и ряды.

**Приятного кодинга!**

### Какие библиотеки мы используем и почему не другие

| Метод | Реализация авторов | На Colab |
|---|---|---|
| Polyjuice | пакет `polyjuice-nlp` 0.1.3, модель `uw-hai/polyjuice` | колесо 0.1.3 на PyPI ставится без модуля `polyjuice` (в нем только метаданные), поэтому модель берем из Hugging Face через `transformers` и формат промпта из кода авторов |
| MiCE | репозиторий `allenai/mice` | пакетом не публикуется и собран на AllenNLP, который больше не развивается; редактор и бинарный поиск пишем сами на `transformers` |
| Замены регионов (Goyal и соавторы) | код при статье | пакета нет; жадный поиск пишем на torch |
| Native-Guide | репозиторий `e-delaney/Instance-Based_CFE_TSC` | написан на TensorFlow и tslearn; классификатор и поиск пишем на torch |
| DiME | репозиторий `guillaumejs2403/DiME` | нужна диффузионная модель, обученная на изображениях лиц или ImageNet, и сотни шагов диффузии на одну картинку; в практике не считаем, остается в теории урока |

Все модели и данные открытые и скачиваются без ключей: DistilBERT SST-2, `t5-small`, `uw-hai/polyjuice` и GPT-2 с Hugging Face, отзывы SST-2 через `datasets`, ResNet-50 из `torchvision`, снимки из папки `data` репозитория курса, ряды GunPoint из архива UCR/UEA. На CPU Colab тетрадь идет около 10 минут, дольше всего дообучение редактора MiCE и генерация Polyjuice; на GPU быстрее. Числа в разделах с обучением и с генерацией могут немного отличаться на другом железе.

In [ ]:
!pip install transformers==5.18.0 datasets==4.8.5 sentencepiece==0.2.2 -q   # в Colab уже стоят; закрепляем версии прогона

In [ ]:
import time, io, zipfile, urllib.request, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torchvision
import torchvision.transforms.functional as TF
from PIL import Image
from IPython.display import display
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoModelForSeq2SeqLM, AutoModelForCausalLM)

device = "cuda" if torch.cuda.is_available() else "cpu"
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_all(0)
pd.set_option("display.max_colwidth", 120)
print("device:", device)

## 1. Текст: классификатор и отзывы

Классификатор — DistilBERT, дообученный на SST-2 (бинарная тональность отзывов о фильмах). Возьмем из валидационной части SST-2 первые 12 отзывов длиной от 6 до 20 слов, на которых классификатор не ошибается: их прогноз мы и будем переворачивать.

In [ ]:
CLF = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
ctok = AutoTokenizer.from_pretrained(CLF)
clf = AutoModelForSequenceClassification.from_pretrained(CLF).to(device).eval()
LAB = ["negative", "positive"]

def probs(texts):                                  # вероятности классов SST-2 для списка текстов
    with torch.no_grad():
        e = ctok(texts, return_tensors="pt", padding=True, truncation=True).to(device)
        return clf(**e).logits.softmax(-1).cpu().numpy()

val = load_dataset("stanfordnlp/sst2", split="validation")
cand = [r for r in val if 6 <= len(r["sentence"].split()) <= 20]
P0 = probs([r["sentence"] for r in cand])
texts = [r["sentence"].strip() for r, pr in zip(cand, P0) if pr.argmax() == r["label"]][:12]
y0 = probs(texts).argmax(1)
pd.DataFrame({"текст": texts, "прогноз": [LAB[k] for k in y0]})

### Пословное расстояние Левенштейна

MiCE меряет минимальность правки пословным расстоянием Левенштейна (врезка в шаге «MiCE — минимальные контрастивные правки» урока): минимальное число вставок, удалений и замен слов, превращающих $x$ в $x'$,

$$\mathrm{lev}(i,j)=\min\big\{\mathrm{lev}(i-1,j)+1,\ \mathrm{lev}(i,j-1)+1,\ \mathrm{lev}(i-1,j-1)+\mathbb{1}[x_i\neq x'_j]\big\},$$

с базой $\mathrm{lev}(i,0)=i$, $\mathrm{lev}(0,j)=j$.

**Задание 1.** Допишите `lev(a, b)` динамикой по строкам: держите одну строку таблицы `D`, на каждом слове $a_i$ обновляйте ее по рекурренте и верните последний элемент.

*Вопрос для самопроверки:* чему равно расстояние для пары из задания урока — «собрала портфолио из трех макетов» и «собрала большое портфолио из пяти макетов»?

<details><summary>Проверить себя</summary>

$\mathrm{lev} = 2$: одна вставка («большое») и одна замена («трех» на «пяти»). Решение: `D = np.arange(len(b) + 1)`, затем для каждого `i` сохранить `prev = D.copy()`, положить `D[0] = i` и для каждого `j` взять `min(prev[j] + 1, D[j - 1] + 1, prev[j - 1] + (a[i - 1] != b[j - 1]))`.
</details>

In [ ]:
def lev(a, b):                                     # пословное расстояние Левенштейна
    a, b = a.split(), b.split()
    ...  # Ваш код здесь: строка динамики D и рекуррента по словам

print(lev("собрала портфолио из трех макетов", "собрала большое портфолио из пяти макетов"))

## 2. MiCE: минимальная контрастивная правка

MiCE (Ross, Marasović, Peters, 2021) решает задачу в два этапа (шаг «MiCE — минимальные контрастивные правки»).

**Этап 1.** Редактор T5 учится восстанавливать замаскированные слова при заданной метке: на вход идет `label: <метка>. input: <текст с масками>`, на выходе — содержимое масок. У авторов маскируют слова с наибольшим градиентом и учат T5-base; мы для скорости берем `t5-small`, случайную маску от 20 до 55% слов и 300 шагов по 16 отзывов из обучающей части SST-2. На CPU это пара минут; результат обучения зависит от железа.

**Этап 2.** Для отзыва с меткой $y$ и контрастной меткой $y_c = 1 - y$ считаем норму градиента потерь $y_c$ по эмбеддингам каждого слова, маскируем долю $n$ слов с наибольшей нормой, просим редактор дописать маску при метке $y_c$ (beam search, 15 кандидатов) и проверяем кандидатов классификатором. Долю $n$ ищем бинарным поиском на $[0, 0.55]$ за 4 шага: если хоть один кандидат меняет прогноз, маску уменьшаем, если нет — увеличиваем. Ответ — кандидат с наименьшим $\mathrm{lev}$ при наименьшей успешной маске.

In [ ]:
ED = "google-t5/t5-small"
ttok = AutoTokenizer.from_pretrained(ED)
editor = AutoModelForSeq2SeqLM.from_pretrained(ED).to(device)

def mask_words(words, idx):                        # слова с номерами idx -> сентинелы T5, ответ — их содержимое
    src, tgt, k, prev = [], [], 0, False
    for i, w in enumerate(words):
        if i in idx:
            if not prev:
                src.append(f"<extra_id_{k}>"); tgt.append(f"<extra_id_{k}>"); k += 1
            tgt.append(w); prev = True
        else:
            src.append(w); prev = False
    tgt.append(f"<extra_id_{k}>")
    return " ".join(src), " ".join(tgt)

train = load_dataset("stanfordnlp/sst2", split="train")
train = train.filter(lambda r: 6 <= len(r["sentence"].split()) <= 30).shuffle(seed=0).select(range(4800))
g = np.random.default_rng(0)
opt = torch.optim.AdamW(editor.parameters(), 3e-4)
editor.train(); t0 = time.time()
for step in range(300):                            # этап 1: редактор учится восстанавливать слова при заданной метке
    batch = train[step * 16:(step + 1) * 16]
    S, T = [], []
    for s, l in zip(batch["sentence"], batch["label"]):
        w = s.split(); k = max(1, int(len(w) * g.uniform(0.2, 0.55)))
        a, b = mask_words(w, set(g.choice(len(w), k, replace=False).tolist()))
        S.append(f"label: {LAB[l]}. input: {a}"); T.append(b)
    e = ttok(S, return_tensors="pt", padding=True).to(device)
    lab = ttok(T, return_tensors="pt", padding=True).input_ids.to(device)
    lab[lab == ttok.pad_token_id] = -100
    loss = editor(**e, labels=lab).loss
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 100 == 0: print("шаг %d, loss %.2f, %.0f с" % (step, loss.item(), time.time() - t0))
editor.eval()
print("этап 1 за %.0f с" % (time.time() - t0))

**Задание 2.** Допишите в `mice` шаг бинарного поиска: если среди кандидатов есть такие, что меняют прогноз (`ok` не пуст), запомните в `best` пару (кандидат с наименьшим `lev` до исходного текста, доля маски `mid`) и сдвиньте верхнюю границу `hi` на `mid`; иначе сдвиньте нижнюю границу `lo` на `mid`.

*Вопрос для самопроверки:* на скольких из 12 отзывов MiCE перевернул прогноз и какие правки получились у самых коротких?

<details><summary>Проверить себя</summary>

В решенной копии MiCE перевернул прогноз на 10 отзывах из 12. Самые короткие правки меняют одно слово при маске 0.034: «charming» → «stale», «slow» → «funny» в конце отзыва «it 's slow -- very , very slow», «taxes» → «best». У длинных правок беглость страдает: «the mesmerizing performances of the leads keep the film grounded…» превратилось в «the length of the leads of the film , and the length of the film» при маске 0.413. Числа зависят от дообучения редактора. Решение: `d = [lev(text, c) for c in ok]; best = (ok[int(np.argmin(d))], mid); hi = mid`, иначе `lo = mid`.
</details>

In [ ]:
def word_grads(text, target):                      # норма градиента по эмбеддингам, максимум по токенам слова
    words = text.split()
    e = ctok(words, is_split_into_words=True, return_tensors="pt").to(device)
    emb = clf.distilbert.embeddings.word_embeddings(e.input_ids).detach().requires_grad_(True)
    out = clf(inputs_embeds=emb, attention_mask=e.attention_mask).logits
    (-out.log_softmax(-1)[0, target]).backward()   # градиент потерь контрастной метки
    gn = emb.grad[0].norm(dim=-1).cpu().numpy()
    s = np.zeros(len(words))
    for t, w in enumerate(e.word_ids()):
        if w is not None: s[w] = max(s[w], gn[t])
    return words, s

def infill(words, idx, yc, n_beams=15):            # T5 дописывает маску при контрастной метке, beam search
    src, _ = mask_words(words, idx)
    k = src.count("<extra_id_")
    ids = ttok(f"label: {LAB[yc]}. input: {src}", return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        gen = editor.generate(ids, num_beams=n_beams, num_return_sequences=n_beams, max_new_tokens=8 * k + 5)
    out = []
    for gseq in gen:
        dec = ttok.decode(gseq, skip_special_tokens=False).replace("<pad>", "").replace("</s>", "")
        txt = src
        for j in range(k):
            a = dec.find(f"<extra_id_{j}>"); b = dec.find(f"<extra_id_{j + 1}>")
            fill = dec[a + len(f"<extra_id_{j}>"): b if b >= 0 else None].strip() if a >= 0 else ""
            txt = txt.replace(f"<extra_id_{j}>", fill)
        out.append(" ".join(txt.split()))
    return out

def mice(text, steps=4, n_max=0.55):
    y = probs([text])[0].argmax(); yc = 1 - y
    words, s = word_grads(text, yc)
    order = np.argsort(-s)                         # сначала слова с наибольшим градиентом
    lo, hi, best, calls = 0.0, n_max, None, 0
    for _ in range(steps):                         # этап 2: бинарный поиск по доле маскируемых слов
        mid = (lo + hi) / 2
        k = max(1, int(round(mid * len(words))))
        cands = infill(words, set(order[:k].tolist()), yc)
        pc = probs(cands)[:, yc]; calls += len(cands)
        ok = [c for c, q in zip(cands, pc) if q > 0.5]
        if ok:
        ...  # Ваш код здесь: обновить best и hi, если ok не пуст, иначе lo
    return y, best, calls

rows = []
t0 = time.time()
for t in texts:
    y, best, calls = mice(t)
    rows.append({"текст": t, "было": LAB[y], "правка MiCE": best[0] if best else None,
                 "доля маски": round(best[1], 3) if best else None,
                 "lev / n": round(lev(t, best[0]) / len(t.split()), 3) if best else None,
                 "p контрастной метки": round(probs([best[0]])[0, 1 - y], 3) if best else None,
                 "вызовов": calls})
mice_df = pd.DataFrame(rows)
print("MiCE: перевернуто %d из %d, %.0f с" % (mice_df["правка MiCE"].notna().sum(), len(texts), time.time() - t0))
mice_df

У MiCE нет отдельного фильтра беглости: беглость держит сам редактор, а `t5-small` после 300 шагов держит ее плохо, когда маска закрывает треть отзыва. Там, где хватает одного-двух слов, правка читается как объяснение: «charming» и «stale» — это и есть слово, которое решило прогноз.

## 3. Polyjuice: беглые варианты без оглядки на классификатор

Polyjuice (Wu, Ribeiro, Heer, Weld, 2021) — дообученный GPT-2, который по исходному тексту, коду правки и тексту с пропуском `[BLANK]` дописывает пропуск. Формат промпта из кода авторов: `<текст> <|perturb|> [<код>] <текст с [BLANK]> [SEP]`, ответ заканчивается `[ANSWER]`. Классификатор генератор не видит: сначала порождаем кандидатов, потом фильтруем.

Мы ставим пропуск на каждое слово отзыва по очереди и для каждого пропуска пробуем четыре кода: `negation`, `lexical`, `quantifier`, `resemantic`. Фильтр беглости из шага урока: GPT-2 считает суммарную лог-вероятность (натуральный логарифм) исходного текста и варианта, и вариант отбрасывают, если она упала больше чем на 10; у авторов есть еще такая же проверка по измененному фрагменту, мы ограничимся всем предложением. Генерация со случайной выборкой (`top_p = 0.9`), сид фиксирован.

In [ ]:
PJ = "uw-hai/polyjuice"
ptok = AutoTokenizer.from_pretrained(PJ)
pj = AutoModelForCausalLM.from_pretrained(PJ).to(device).eval()
gtok = AutoTokenizer.from_pretrained("openai-community/gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("openai-community/gpt2").to(device).eval()
CODES = ["negation", "lexical", "quantifier", "resemantic"]

def polyjuice(text, code, i):                      # промпт в формате авторов: текст, код правки, текст с [BLANK]
    w = text.split(); blanked = " ".join(w[:i] + ["[BLANK]"] + w[i + 1:])
    prompt = f"{text} <|perturb|> [{code}] {blanked} [SEP]"
    ids = ptok(prompt, return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        out = pj.generate(ids, do_sample=True, top_p=0.9, max_new_tokens=12, pad_token_id=ptok.eos_token_id)
    fill = ptok.decode(out[0, ids.shape[1]:]).split("[ANSWER]")[0].replace("<|endoftext|>", "").strip()
    if fill == "EMPTY": fill = ""
    return " ".join(blanked.replace("[BLANK]", fill).split())

def logprob(texts):                                # суммарная лог-вероятность текста у GPT-2 (натуральный логарифм)
    res = []
    for t in texts:
        ids = gtok(t, return_tensors="pt").input_ids.to(device)
        with torch.no_grad(): res.append(-gpt2(ids, labels=ids).loss.item() * (ids.shape[1] - 1))
    return np.array(res)

seed_all(0); t0 = time.time()
rows = []
for t in texts:
    y = probs([t])[0].argmax(); lp0 = logprob([t])[0]
    for i, w in enumerate(t.split()):
        if not any(ch.isalpha() for ch in w): continue    # знаки препинания не заменяем
        for code in CODES:
            v = polyjuice(t, code, i)
            if v == t: continue
            rows.append({"текст": t, "было": y, "код": code, "вариант": v, "lp0": lp0})
pj_df = pd.DataFrame(rows).drop_duplicates(["текст", "вариант"])
pj_df["Δ лог-вероятности"] = logprob(pj_df["вариант"].tolist()) - pj_df["lp0"]
pj_df["p контрастной метки"] = [probs([v])[0, 1 - y] for v, y in zip(pj_df["вариант"], pj_df["было"])]
pj_df["lev / n"] = [lev(t, v) / len(t.split()) for t, v in zip(pj_df["текст"], pj_df["вариант"])]
print("Polyjuice: %d вариантов за %.0f с" % (len(pj_df), time.time() - t0))

**Задание 3.** Оставьте в `fluent` беглые варианты (Δ лог-вероятности не ниже −10), а в `flipped` — те из них, у которых вероятность контрастной метки больше 0.5.

*Вопрос для самопроверки:* сколько вариантов прошли фильтр беглости, сколько из них меняют прогноз и на скольких отзывах Polyjuice нашел хотя бы один контрфактуал?

<details><summary>Проверить себя</summary>

В решенной копии из 456 вариантов беглых 247, прогноз меняют 16, и контрфактуал нашелся на 7 отзывах из 12. Чаще всего прогноз меняют коды `lexical` (8) и `negation` (5). Числа зависят от случайной генерации. Решение: `fluent = pj_df[pj_df["Δ лог-вероятности"] >= -10]`, `flipped = fluent[fluent["p контрастной метки"] > 0.5]`.
</details>

In [ ]:
# Задание 3: оставьте беглые варианты (Δ лог-вероятности не ниже -10) и из них те, что меняют прогноз
fluent = ...   # Ваш код здесь
flipped = ...  # Ваш код здесь
print("вариантов: %d, беглых: %d, из них меняют прогноз: %d" % (len(pj_df), len(fluent), len(flipped)))
print("текстов, где Polyjuice нашел хотя бы один контрфактуал: %d из %d" % (flipped["текст"].nunique(), len(texts)))
display(flipped.groupby("код").size().rename("меняют прогноз").to_frame()
        .join(fluent.groupby("код").size().rename("беглых")))
best_pj = flipped.sort_values("lev / n").groupby("текст").head(1)
best_pj[["текст", "код", "вариант", "lev / n", "p контрастной метки"]]

Сведем оба метода по отзывам: перевернул ли метод прогноз и какая доля слов изменилась ($\mathrm{lev}/n$).

In [ ]:
cmp = pd.DataFrame({"текст": texts,
                    "MiCE: lev / n": mice_df["lev / n"].values,
                    "Polyjuice: лучший lev / n": [best_pj.set_index("текст")["lev / n"].get(t, np.nan) for t in texts]})
print("доля перевернутых текстов: MiCE %.2f, Polyjuice %.2f"
      % (cmp["MiCE: lev / n"].notna().mean(), cmp["Polyjuice: лучший lev / n"].notna().mean()))
print("медиана lev / n там, где нашли оба: MiCE %.3f, Polyjuice %.3f"
      % tuple(cmp.dropna()[["MiCE: lev / n", "Polyjuice: лучший lev / n"]].median()))
cmp

В решенной копии MiCE перевернул 10 отзывов из 12, Polyjuice 7 из 12, но там, где нашли оба, правка Polyjuice короче: медиана $\mathrm{lev}/n$ 0.058 против 0.200. Это два полюса из вывода урока. MiCE целенаправленный: градиент указывает, где править, и редактор пишет под контрастную метку, поэтому он находит правку чаще, но платит длиной и беглостью. Polyjuice перебирает однословные правки, не глядя на модель: правки короткие и беглые, но прогноз меняет только малая доля из них.

Среди найденных Polyjuice есть и такие, что меняют прогноз, не меняя смысла: «you do n't have to know about music to appreciate the film…» с заменой «know» на «care» классификатор уже считает отрицательным отзывом. Такой контрфактуал объясняет не отзыв, а уязвимость классификатора, и ради этого Polyjuice и используют при отладке моделей.

## 4. Изображения: замены регионов (Goyal и соавторы)

Goyal и соавторы (2019) берут запрос $I$ класса $c$ и дистрактор $I'$ целевого класса $c'$ и ищут на признаковой карте наименьшее число ячеек, замена которых ячейками дистрактора переворачивает решение (шаг «Контрфактуалы на изображениях: регионы»):

$$\min_{a,\,P}\ \lVert a\rVert_1\quad\text{при}\quad c'=\arg\max\ g\big((1-a)\odot f(I) + a\odot P\,f(I')\big).$$

У ResNet-50 карта $f$ после последнего блока — $7 \times 7$ ячеек по 2048 признаков, а $g$ — среднее по ячейкам и линейный слой. Среднее линейно, поэтому логиты после замены ячейки $i$ запроса ячейкой $j$ дистрактора считаются для всех $49 \cdot 49$ пар сразу, без прогонов сети. Жадный поиск на каждом шаге берет пару с наибольшей log-вероятностью $c'$; каждая ячейка запроса и дистрактора используется один раз, потому что $P$ — перестановка.

Снимки берем из папки `data` курса. Запрос — `cat.jpg`, ResNet-50 относит его к классу tiger cat (полосатая домашняя кошка); дистрактор — `cat_piran.jpg`, класс Egyptian cat (египетская кошка): близкие классы, как пары видов птиц у авторов. Веса `IMAGENET1K_V2` учились со сглаживанием меток, поэтому вероятности верхнего класса у них невысокие.

Автор фото кошки: [Paul Hanaoka, Unsplash](https://unsplash.com/photos/w2DsS-ZAP4U), лицензия Unsplash.

Автор фото кошки из Пирана (`cat_piran.jpg`): Jakub Hałun, [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Cat_in_Piran,_Slovenia,_20240504_1600_8594.jpg), лицензия [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

Автор фото хрюшки: [Brett Sayles, Pexels](https://www.pexels.com/photo/4966539/), лицензия Pexels.

Автор фото кошки и мопса: [Priscilla Du Preez, Unsplash](https://unsplash.com/photos/qtkpOxCpl0Y), лицензия Unsplash.

Автор фото зебр (`geranimo-ADUi-P4n-Jwds-unsplash.jpg`): [Geranimo, Unsplash](https://unsplash.com/photos/ADUiP4nJwds), лицензия Unsplash.

In [ ]:
DATA = "https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data"
def load_img(name): return Image.open(io.BytesIO(urllib.request.urlopen(f"{DATA}/{name}", timeout=30).read())).convert("RGB")

W50 = torchvision.models.ResNet50_Weights.IMAGENET1K_V2
resnet = torchvision.models.resnet50(weights=W50).to(device).eval()
prep, CATS = W50.transforms(), W50.meta["categories"]
body = nn.Sequential(*list(resnet.children())[:-2])   # до layer4 включительно: карта 2048 x 7 x 7
Wfc, bfc = resnet.fc.weight.detach(), resnet.fc.bias.detach()

def cells(img):                                    # признаковая карта как 49 ячеек по 2048 признаков
    with torch.no_grad(): return body(prep(img)[None].to(device))[0].flatten(1).T
def head(F): return F.mean(0) @ Wfc.T + bfc        # g: среднее по ячейкам и линейный слой fc

imgs = {n: load_img(n) for n in ["cat.jpg", "cat_piran.jpg", "geranimo-ADUi-P4n-Jwds-unsplash.jpg", "piglet.jpg", "cat_and_dog.jpg"]}
F = {n: cells(im) for n, im in imgs.items()}
pd.DataFrame({n: {"класс": CATS[head(f).argmax().item()], "p": round(head(f).softmax(0).max().item(), 3)} for n, f in F.items()}).T

**Задание 4.** Допишите в `goyal` жадный шаг: найдите пару $(i, j)$ с наибольшим `s` (`s` — матрица $49 \times 49$, номер пары — `divmod(s.argmax().item(), 49)`), замените ячейку `Fc[i]` ячейкой `Fd[j]` и добавьте пару в `edits`.

*Вопрос для самопроверки:* сколько замен понадобилось, чтобы полосатая кошка стала египетской, и где на снимках лежат эти ячейки?

<details><summary>Проверить себя</summary>

Хватило двух замен: ячейки 36 и 29 запроса (строки 5 и 4, столбец 1, стол и кувшин слева от кошки, а не сама кошка) на ячейки 27 и 26 дистрактора (строка 3, столбцы 6 и 5, морда серой кошки), $p(\text{Egyptian cat}) = 0.314$. Решение: `i, j = divmod(s.argmax().item(), 49)`, `Fc[i] = Fd[j]`, `edits.append((i, j))`.
</details>

In [ ]:
def goyal(Fq, Fd):
    c, cd = head(Fq).argmax().item(), head(Fd).argmax().item()
    Fc, edits = Fq.clone(), []
    while head(Fc).argmax().item() != cd:
        # логиты после замены ячейки i запроса ячейкой j дистрактора: среднее линейно, поэтому считаем сразу все 49 x 49 пар
        L = head(Fc)[None, None, :] + ((Fd[None, :, :] - Fc[:, None, :]) / 49) @ Wfc.T
        s = L.log_softmax(-1)[..., cd]             # log-вероятность класса дистрактора
        for i, j in edits: s[i] = -np.inf; s[:, j] = -np.inf   # P — перестановка: каждую ячейку берем один раз
        # Задание 4: выберите пару (i, j) с наибольшим s, замените ячейку i запроса ячейкой j дистрактора и запишите пару в edits
        ...  # Ваш код здесь: пара (i, j) с наибольшим s, замена Fc[i] = Fd[j], запись в edits
    return CATS[c], CATS[cd], edits, head(Fc).softmax(0)[cd].item()

q, d = "cat.jpg", "cat_piran.jpg"
c, cd, edits, pcd = goyal(F[q], F[d])
print("%s -> %s: замен %d из 49, пары (ячейка запроса, ячейка дистрактора) %s, p(%s) = %.3f" % (c, cd, len(edits), edits, cd, pcd))

In [ ]:
def show_cells(ax, img, idx, color, title):
    im = TF.center_crop(TF.resize(img, prep.resize_size[0]), prep.crop_size[0])   # ровно тот кадр, который видит модель
    ax.imshow(im)
    for k in idx:
        r, cc = divmod(k, 7)
        ax.add_patch(plt.Rectangle((cc * 32, r * 32), 32, 32, fill=False, ec=color, lw=3))
    ax.set_title(title, fontsize=10); ax.axis("off")

fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
show_cells(ax[0], imgs[q], [i for i, _ in edits], "#eb6834", f"запрос: {c}, заменяемая ячейка")
show_cells(ax[1], imgs[d], [j for _, j in edits], "#1baf7a", f"дистрактор: {cd}, ячейка-источник")
plt.tight_layout(); plt.show()

На рисунке слева кадр запроса так, как его видит модель после обрезки, оранжевыми рамками — заменяемые ячейки; справа кадр дистрактора, зелеными рамками — ячейки, откуда взяты признаки. Модели оказалось достаточно увидеть морду серой кошки на месте двух ячеек стола слева от полосатой кошки: заменили не саму кошку, а фон рядом с ней.

Проверим, сколько замен нужно другим парам и насколько жадный выбор лучше случайного порядка пар.

In [ ]:
def random_edits(Fq, Fd, seed):                    # та же замена, но порядок пар случайный
    g = np.random.default_rng(seed); cd = head(Fd).argmax().item()
    Fc = Fq.clone()
    for n, (i, j) in enumerate(zip(g.permutation(49), g.permutation(49)), 1):
        Fc[i] = Fd[j]
        if head(Fc).argmax().item() == cd: return n
    return 49

rows = []
for q_, d_ in [("cat.jpg", "cat_piran.jpg"), ("cat_piran.jpg", "cat.jpg"), ("geranimo-ADUi-P4n-Jwds-unsplash.jpg", "piglet.jpg"),
               ("cat.jpg", "cat_and_dog.jpg")]:
    c_, cd_, ed_, p_ = goyal(F[q_], F[d_])
    rnd = [random_edits(F[q_], F[d_], s) for s in range(20)]
    rows.append({"запрос": f"{q_} ({c_})", "дистрактор": f"{d_} ({cd_})", "замен жадно": len(ed_),
                 "замен случайно, медиана из 20": float(np.median(rnd)), "p целевого класса": round(p_, 3)})
pd.DataFrame(rows)

Жадный поиск в решенной копии дает 2 замены для пары «полосатая кошка → египетская кошка», 7 для «зебра → домашняя свинья» и 14 для «полосатая кошка → мопс», а случайный порядок в медиане 6, 19.5 и 43.5 замены. Обратная пара «египетская кошка → полосатая кошка» несимметрична прямой: жадный поиск заменяет 15 ячеек из 49, случайный порядок — 40.5 в медиане. Правка на треть кадра — уже не малая разреженная правка, и как объяснение она читается с трудом. Есть и вторая слабость метода: ответ зависит от выбранного дистрактора, а дистрактор выбирают вручную.

## 5. Временные ряды: Native-Guide

Native-Guide (Delaney и соавторы, 2021) переписывает в ряду один подотрезок значениями ближайшего ряда другого класса (шаг «Временные ряды: Native-Guide»):

$$x^{CF} = (1 - m)\odot x + m\odot \mathrm{NUN},\qquad \min|m|\ \ \text{при}\ \ g(x^{CF}) = c'.$$

Шаги: найти NUN (native unlike neighbour) — ближайший обучающий ряд целевого класса $c'$; по CAM классификатора на NUN найти окно с наибольшей суммой важности; растить длину окна, пока прогноз не сменится.

Набор GunPoint (Ratanamahatana и Keogh) из архива UCR Time Series Classification Archive (Dau и соавторы, 2018, [arXiv:1810.07758](https://arxiv.org/abs/1810.07758)); тетрадь скачивает его с сайта архива [timeseriesclassification.com](https://www.timeseriesclassification.com/description.php?Dataset=GunPoint), а в папку `data` курса не копирует: условий, разрешающих распространять данные, у архива нет, он просит только ссылаться на статью. Набор: координата руки актера во времени, 150 отсчетов; класс Gun — актер достает муляж пистолета из кобуры, целится и убирает обратно, класс Point — просто указывает пальцем. 50 рядов в обучении, 150 в тесте. CAM нужен классификатору с глобальным усреднением, поэтому берем FCN Wang и соавторов (2017): три свертки, усреднение по времени и линейный слой, 300 эпох Adam на всей обучающей выборке.

In [ ]:
# GunPoint с сайта архива UCR/UEA (Dau и соавторы, 2018): условий распространения у архива нет, поэтому файлов нет в data/ курса
raw = urllib.request.urlopen("https://www.timeseriesclassification.com/aeon-toolkit/GunPoint.zip", timeout=60).read()
zf = zipfile.ZipFile(io.BytesIO(raw))
tr = np.loadtxt(io.BytesIO(zf.read("GunPoint_TRAIN.txt"))); te = np.loadtxt(io.BytesIO(zf.read("GunPoint_TEST.txt")))
Xtr, ytr = tr[:, 1:].astype(np.float32), (tr[:, 0] - 1).astype(int)    # классы 0 = Gun, 1 = Point
Xte, yte = te[:, 1:].astype(np.float32), (te[:, 0] - 1).astype(int)
print("обучение", Xtr.shape, "тест", Xte.shape, "классы в обучении", np.bincount(ytr))

class FCN(nn.Module):                              # FCN Wang и соавторов: три свертки, глобальное усреднение, линейный слой
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(nn.Conv1d(1, 32, 7, padding=3), nn.BatchNorm1d(32), nn.ReLU(),
                               nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(),
                               nn.Conv1d(64, 32, 3, padding=1), nn.BatchNorm1d(32), nn.ReLU())
        self.fc = nn.Linear(32, 2)
    def forward(self, x): return self.fc(self.f(x[:, None]).mean(-1))
    def cam(self, x, c): return torch.einsum("k,bkt->bt", self.fc.weight[c], self.f(x[:, None]))   # CAM вдоль времени

seed_all(0)
fcn = FCN().to(device)
opt = torch.optim.Adam(fcn.parameters(), 1e-2)
Xt, yt = torch.tensor(Xtr, device=device), torch.tensor(ytr, device=device)
t0 = time.time()
for epoch in range(300):
    fcn.train(); loss = nn.functional.cross_entropy(fcn(Xt), yt)
    opt.zero_grad(); loss.backward(); opt.step()
fcn.eval()
def ts_probs(X):
    with torch.no_grad(): return fcn(torch.tensor(np.asarray(X, np.float32), device=device)).softmax(1).cpu().numpy()
print("FCN за %.0f с, точность на тесте %.3f" % (time.time() - t0, (ts_probs(Xte).argmax(1) == yte).mean()))

**Задание 5.** Допишите в `native_guide` поиск NUN: среди рядов `pool` (обучающие ряды целевого класса, которые модель к нему и относит) возьмите ближайший к `x` по евклидову расстоянию.

*Вопрос для самопроверки:* какой отрезок пришлось переписать в тестовом ряду 1 и какую долю ряда он занимает?

<details><summary>Проверить себя</summary>

В решенной копии ряд 1 класса Point стал рядом класса Gun после замены отрезка $[81, 108)$, 27 отсчетов из 150, то есть 18% ряда. Отрезок приходится на опускание руки, где у класса Gun рука убирает пистолет в кобуру. Числа зависят от обучения FCN. Решение: `nun = pool[np.argmin(((pool - x)**2).sum(1))]`.
</details>

In [ ]:
def native_guide(x):
    c = ts_probs([x])[0].argmax(); cp = 1 - c
    pool = Xtr[(ytr == cp) & (ts_probs(Xtr).argmax(1) == cp)]   # обучающие ряды целевого класса, которые модель так и относит
    # Задание 5: NUN — ближайший к x ряд из pool по евклидову расстоянию
    nun = ...  # Ваш код здесь: ближайший к x ряд из pool
    with torch.no_grad(): w = fcn.cam(torch.tensor(nun[None], device=device), cp)[0].cpu().numpy()
    for L in range(1, len(x) + 1):                 # растим длину подотрезка, пока прогноз не сменится
        s = int(np.argmax(np.convolve(w, np.ones(L), "valid")))   # окно длины L с наибольшей суммой CAM у NUN
        xc = x.copy(); xc[s:s + L] = nun[s:s + L]
        if ts_probs([xc])[0, cp] > 0.5: return xc, nun, s, L, c

x = Xte[1]
xc, nun, s, L, c = native_guide(x)
print("ряд 1: класс %d -> %d, переписан отрезок [%d, %d), длина %d из %d" % (c, 1 - c, s, s + L, L, len(x)))
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(x, color="black", lw=1.5, label="исходный ряд")
ax.plot(nun, color="#1baf7a", lw=1, ls="--", label="NUN — ближайший ряд другого класса")
ax.plot(np.arange(s, s + L), xc[s:s + L], color="#eb6834", lw=3, label="переписанный подотрезок")
ax.axvspan(s, s + L, color="#eb6834", alpha=0.1)
ax.set_xlabel("время, отсчеты"); ax.set_ylabel("координата руки"); ax.legend(fontsize=8)
ax.set_title("Native-Guide на GunPoint"); plt.show()

На рисунке черным — исходный ряд, зеленым пунктиром — NUN, оранжевым — переписанный подотрезок; остальной ряд не тронут, а вставленный кусок взят из реального ряда, поэтому сигнал остается гладким.

Посчитаем Native-Guide для всех 150 тестовых рядов и проверим, важен ли выбор места по CAM: перенесем отрезок той же длины от того же NUN в 20 случайных мест ряда.

In [ ]:
def random_segment(x, nun, L, cp, g):              # тот же сосед и та же длина, но отрезок в случайном месте
    s = g.integers(0, len(x) - L + 1); xc = x.copy(); xc[s:s + L] = nun[s:s + L]
    return ts_probs([xc])[0, cp] > 0.5

g = np.random.default_rng(0); rows = []
for i in range(len(Xte)):
    xc, nun, s, L, c = native_guide(Xte[i])
    rows.append({"ряд": i, "класс": c, "длина": L, "доля ряда": L / Xte.shape[1],
                 "случайное место той же длины": np.mean([random_segment(Xte[i], nun, L, 1 - c, g) for _ in range(20)])})
ng = pd.DataFrame(rows)
print("медиана длины переписанного отрезка: %d из %d отсчетов" % (ng["длина"].median(), Xte.shape[1]))
print("медиана длины по исходному классу:", ng.groupby("класс")["длина"].median().to_dict())
print("доля рядов, где хватило 20%% длины: %.2f" % (ng["доля ряда"] <= 0.2).mean())
print("тот же отрезок в случайном месте меняет прогноз в %.2f случаев" % ng["случайное место той же длины"].mean())
ng.groupby("класс")["длина"].describe()

В решенной копии медиана длины переписанного отрезка — 38 отсчетов из 150, и только в 39% рядов хватило не больше 20% длины. Метод несимметричен: чтобы ряд класса Point стал рядом класса Gun, хватает короткого отрезка на опускании руки, а для обратного перехода переписывают больше половины ряда, потому что признаки Gun у FCN занимают большую часть записи. Место по CAM важно: тот же отрезок от того же NUN в случайном месте меняет прогноз в 28% случаев. Native-Guide опирается и на CAM, и на соседа из выборки; если ближайший ряд другого класса далек от запроса, отрезок становится длинным, и объяснение перестает быть разреженным.

## 6. Почему DiME остается в теории

DiME (Jeanneret, Simon, Jurie, 2022) зашумляет картинку диффузией и на обратном проходе подталкивает ее градиентом классификатора к целевому классу. Для этого нужна диффузионная модель, обученная на той же области, что и классификатор (у авторов — лица CelebA и ImageNet), и сотни шагов денойзинга с градиентом классификатора на одну картинку. Готовой модели такого масштаба, которую можно запустить в бесплатном Colab за минуты, нет, поэтому метод разобран в уроке и в практике не считается; то же верно для его преемников ACE и FastDiME из шага «Вывод: одна постановка на все модальности».

## **Выводы**

В этой практике мы проверили на трех модальностях, что контрфактуальная постановка меняет только операцию правки и модель правдоподобности:
- в тексте минимальность меряют пословным расстоянием Левенштейна; MiCE находит правку чаще (10 отзывов из 12), потому что градиент указывает место, а редактор пишет под контрастную метку, но длинные правки теряют беглость;
- Polyjuice не смотрит на классификатор: однословные правки короткие и беглые, прогноз меняет малая их доля, и среди них встречаются правки, которые меняют прогноз без смены смысла, то есть находят уязвимость модели;
- замены регионов дают короткое и наглядное объяснение, когда классы близки и признаки сосредоточены в малой области, и вырождаются в замену всего кадра, когда признаки класса рассыпаны по нему; жадный выбор пар не всегда лучше случайного;
- Native-Guide переписывает отрезок реального соседа и держит ряд гладким, а длина отрезка зависит от направления перехода и от того, насколько близок сосед; место по CAM важнее, чем случайное место той же длины;
- DiME и его преемники требуют диффузионной модели и остаются в теории урока.